# Notebook 18f: Train Degree Signature Neural Network

## Purpose
Train a neural network to predict pathway counts for DEGREE BIN COMBINATIONS,
not individual node pairs. This reduces memory requirements by 1000x while
capturing essential degree-structure relationships.

## Method
Uses degree-binned training data where nodes are grouped into 10x10 degree bins.
Training set contains ~100 rows (one per source_bin, target_bin combination).

For each bin combination, the model learns:
- Source degree bin → expected pathway count
- Target degree bin → expected pathway count
- Intermediate node degree signature (10x10 histogram) → pathway count modulation

**Important**: Predictions are for AGGREGATED degree bins, not individual pairs.
Individual pair predictions require looking up the bin and applying the model.

## Model Architecture
- **Input**: 102 features (source_bin, target_bin, inter_sig_0...inter_sig_99)
- **Hidden layers**: (128, 64, 32) with ReLU activation and Dropout (0.1)
- **Output**: Expected pathway count for a degree bin combination
- **Training**: Adam optimizer, MSE loss, early stopping (patience=50)
- **Expected performance**: Pearson r ≈ 0.85-0.95 on held-out degree bins

## Training Data Structure
- **Rows**: ~100 (10 source bins × 10 target bins)
- **Features**: 102 (2 degree bins + 100 intermediate signature dimensions)
- **Target**: Mean pathway count across all node pairs in that bin combination

## Inputs
- `results/pathway_nn/training_data/{metapath}_training_data.csv`
  - Generated by notebook 18a (data preparation)
  - Contains degree-binned features and pathway count statistics

## Outputs
- `results/pathway_nn/trained_models/{metapath}_Degree_Sig_NN.pt`
  - Trained PyTorch model (can be loaded for prediction)
- `results/pathway_nn/benchmarks/{metapath}_Degree_Sig_NN_benchmark.json`
  - Training time, memory usage, validation metrics
- `results/pathway_nn/visualizations/{metapath}_Degree_Sig_NN_validation.png`
  - Scatter plot, residual plot, distribution comparison
- `results/pathway_nn/intermediate/{metapath}_test_*.npy`
  - Test predictions, actuals, features (for reproducibility)

## Usage
```bash
# Local execution
jupyter nbconvert --execute notebooks/18f_train_degree_signature_nn.ipynb

# HPC execution with papermill
papermill notebooks/18f_train_degree_signature_nn.ipynb     notebooks/executed/18f_train_degree_signature_nn_executed.ipynb     -p metapath "CbGpPW"
```

## References
- Himmelstein et al. (2017). Systematic integration of biomedical knowledge
  prioritizes drugs for repurposing. eLife. https://doi.org/10.7554/eLife.26726

In [ ]:
# Papermill parameters
metapath = 'CbGpPW'
record_benchmarks = True
random_seed = 42

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.stats import pearsonr
import sys

repo_dir = Path.cwd().parent
sys.path.insert(0, str(repo_dir))

from src.models.degree_signature_nn import DegreeSignatureNN
from src.benchmarking import ModelBenchmarker

print(f"Training Degree Signature NN for {metapath}")

In [ ]:
# Create output directories
model_dir = repo_dir / 'results' / 'pathway_nn' / 'trained_models'
benchmark_dir = repo_dir / 'results' / 'pathway_nn' / 'benchmarks'
model_dir.mkdir(parents=True, exist_ok=True)
benchmark_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Load training data
data_file = repo_dir / 'results' / 'pathway_nn' / 'training_data' / f'{metapath}_training_data.csv'
df = pd.read_csv(data_file)

print(f"Training data: {df.shape}")
print(f"  Pathway count mean range: [{df['pathway_count_mean'].min():.2f}, {df['pathway_count_mean'].max():.2f}]")

In [ ]:
# Prepare features and target
feature_cols = [c for c in df.columns if c.startswith('inter_sig_')]
X = df[['source_bin', 'target_bin'] + feature_cols].values.astype(np.float32)
y = df['pathway_count_mean'].values.astype(np.float32)

print(f"Features: {X.shape}")
print(f"Target: {y.shape}")

In [ ]:
# Split train/test (90/10 due to small dataset size)
# With ~100 degree bin combinations, this gives ~90 train / ~10 test
n_train = int(0.9 * len(X))
indices = np.random.RandomState(random_seed).permutation(len(X))
train_idx = indices[:n_train]
test_idx = indices[n_train:]

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print(f"Dataset size: {len(X)} degree bin combinations")
print(f"Train: {len(X_train)} bins ({100*len(X_train)/len(X):.1f}%)")
print(f"Test: {len(X_test)} bins ({100*len(X_test)/len(X):.1f}%)")
print(f"\nNote: Predictions are for DEGREE BIN COMBINATIONS, not individual pairs")

## Train Model with Benchmarking

In [ ]:
# Initialize benchmarker
benchmarker = ModelBenchmarker(
    model_name='Degree_Sig_NN',
    metapath=metapath,
    n_training_samples=len(X_train),
    n_cores=1
)

# Train with timing
model = DegreeSignatureNN(
    hidden_dims=(128, 64, 32),
    dropout=0.1,
    learning_rate=0.001,
    batch_size=16,
    n_epochs=1000,
    early_stopping_patience=50,
    random_state=random_seed,
    device='cpu'  # Use CPU for HPC
)

print("Training neural network...")
with benchmarker.time_training():
    model.fit(X_train, y_train)

print(f"\nModel trained: {model}")
print(f"Parameters: {model.get_params()}")

## Evaluate and Benchmark

In [ ]:
# Predict with timing
with benchmarker.time_prediction():
    predictions = model.predict(X_test)

# Record validation metrics
benchmarker.record_validation(predictions, y_test)

print(f"\nValidation Results:")
print(f"  Pearson r: {benchmarker.validation_r:.4f}")
print(f"  MAE: {benchmarker.validation_mae:.4f}")
print(f"  RMSE: {benchmarker.validation_rmse:.4f}")

In [ ]:
# Save intermediate outputs for reproducibility
intermediate_dir = repo_dir / 'results' / 'pathway_nn' / 'intermediate'
intermediate_dir.mkdir(parents=True, exist_ok=True)

# Save test predictions
pred_file = intermediate_dir / f'{metapath}_test_predictions.npy'
np.save(pred_file, predictions)
print(f"✓ Saved test predictions: {pred_file}")

# Save test actuals
actual_file = intermediate_dir / f'{metapath}_test_actuals.npy'
np.save(actual_file, y_test)
print(f"✓ Saved test actuals: {actual_file}")

# Save test features
features_file = intermediate_dir / f'{metapath}_test_features.npy'
np.save(features_file, X_test)
print(f"✓ Saved test features: {features_file}")

print(f"\nIntermediate outputs saved to: {intermediate_dir}")

In [ ]:
# Print detailed validation statistics
# Compute R-squared
r2_score = 1 - ((predictions - y_test)**2).sum() / ((y_test - y_test.mean())**2).sum()

print(f"\nDetailed Validation Statistics:")
print(f"{'='*70}")
print(f"Model Type: DEGREE-BINNED (not individual pairs)")
print(f"Test Set Size: {len(y_test)} degree bin combinations")
print(f"\nActual Pathway Counts (per bin):")
print(f"  Mean: {y_test.mean():.4f}")
print(f"  Std:  {y_test.std():.4f}")
print(f"  Min:  {y_test.min():.4f}")
print(f"  Max:  {y_test.max():.4f}")
print(f"\nPredicted Pathway Counts (per bin):")
print(f"  Mean: {predictions.mean():.4f}")
print(f"  Std:  {predictions.std():.4f}")
print(f"  Min:  {predictions.min():.4f}")
print(f"  Max:  {predictions.max():.4f}")
print(f"\nValidation Metrics:")
print(f"  Pearson r:        {benchmarker.validation_r:.4f}")
print(f"  Mean Abs Error:   {benchmarker.validation_mae:.4f} pathways/bin")
print(f"  Root Mean Sq Err: {benchmarker.validation_rmse:.4f} pathways/bin")
print(f"  R² Score:         {r2_score:.4f}")
print(f"\nInterpretation:")
print(f"  - High correlation (r > 0.85) indicates model captures degree effects")
print(f"  - Small test set ({len(y_test)} bins) may show high variance in metrics")
print(f"  - Binned approach reduces memory but loses individual pair resolution")
print(f"{'='*70}")

In [ ]:
# Create figure with 3 subplots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Scatter plot: Predicted vs Actual
axes[0].scatter(y_test, predictions, alpha=0.6, s=50, edgecolors='k',
                linewidth=0.5)
axes[0].plot([y_test.min(), y_test.max()],
             [y_test.min(), y_test.max()],
             'r--', lw=2, label='Perfect prediction')
axes[0].set_xlabel('Actual Pathway Count (Mean)', fontsize=12)
axes[0].set_ylabel('Predicted Pathway Count', fontsize=12)
axes[0].set_title(
    f'Predicted vs Actual (Degree Bins)\nr = {benchmarker.validation_r:.4f}, n = {len(y_test)} bins',
    fontsize=13, fontweight='bold'
)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2. Residual plot
residuals = predictions - y_test
axes[1].scatter(predictions, residuals, alpha=0.6, s=50, edgecolors='k',
                linewidth=0.5)
axes[1].axhline(y=0, color='r', linestyle='--', lw=2)
axes[1].set_xlabel('Predicted Pathway Count', fontsize=12)
axes[1].set_ylabel('Residual (Predicted - Actual)', fontsize=12)
axes[1].set_title(
    f'Residual Plot\nMAE = {benchmarker.validation_mae:.4f}',
    fontsize=13, fontweight='bold'
)
axes[1].grid(True, alpha=0.3)

# 3. Distribution comparison
axes[2].hist(y_test, bins=20, alpha=0.5, label='Actual', color='blue',
             edgecolor='black')
axes[2].hist(predictions, bins=20, alpha=0.5, label='Predicted', color='red',
             edgecolor='black')
axes[2].set_xlabel('Pathway Count', fontsize=12)
axes[2].set_ylabel('Frequency', fontsize=12)
axes[2].set_title(
    f'Distribution Comparison (Degree Bins)\nRMSE = {benchmarker.validation_rmse:.4f}',
    fontsize=13, fontweight='bold'
)
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()

# Save plot
plot_dir = repo_dir / 'results' / 'pathway_nn' / 'visualizations'
plot_dir.mkdir(parents=True, exist_ok=True)
plot_file = plot_dir / f'{metapath}_Degree_Sig_NN_validation.png'
plt.savefig(plot_file, dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Saved visualization: {plot_file}")

## Visualization: Predicted vs Actual Pathway Counts

## Save Model and Benchmark

In [ ]:
# Save model (PyTorch model uses .pt extension)
model_file = model_dir / f'{metapath}_Degree_Sig_NN.pt'
model.save(model_file)
print(f"✓ Saved model: {model_file}")

# Save benchmark
if record_benchmarks:
    result = benchmarker.finalize()
    benchmark_file = benchmark_dir / f'{metapath}_Degree_Sig_NN_benchmark.json'
    result.save_json(benchmark_file)
    print(f"✓ Saved benchmark: {benchmark_file}")
    print(f"\nBenchmark Summary:")
    print(f"  Training time: {result.training_time_sec:.2f}s ({result.training_time_sec/60:.1f} min)")
    print(f"  Prediction time: {result.prediction_time_sec:.2f}s")
    print(f"  Peak memory: {result.peak_memory_gb:.2f} GB")
    print(f"  Normalized cost: {result.normalized_cost:.4f} node-hours")

print(f"\n{'='*70}")
print(f"TRAINING COMPLETE: {metapath} Degree Signature NN")
print(f"{'='*70}")